## Simulation

In [1]:
from nanover.openmm import OpenMMSimulation
from nanover.mdanalysis.converter import frame_data_to_mdanalysis

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()
universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

In [2]:
residue = universe.residues[2]
indices = [int(i) for i in residue.atoms.indices]
calpha = int(residue.atoms.select_atoms("name CA")[0].index)
calpha

23

## iMD runner + client

In [3]:
from nanover.app import OmniRunner
from nanover.websocket import NanoverImdClient

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="MOVEABLE RESTRAINTS")
imd_runner.load(0)
imd = imd_runner.app_server.imd

client = NanoverImdClient.from_runner(imd_runner)

## Restraints

In [4]:
from nanover.app import RenderingSelection
from nanover.imd.imd_state import INTERACTION_PREFIX, ParticleInteraction

RESTRAINT_PREFIX = f"{INTERACTION_PREFIX}MOVEABLE-RESTRAINT."

MOVEABLE_RESTRAINTS: dict[str, ParticleInteraction] = {}
SELECTIONS: dict[str, RenderingSelection] = {}
ACTIVE_RESTRAINTS: set[str] = set()

RENDERER = {
    "render": {
        "particle.scale": .25,
        "bond.scale": 0.0,
        "type": "ball and stick"
    },
    "color": "CornflowerBlue",
}


def add_restraint(indexes):
    key = f"{RESTRAINT_PREFIX}{len(MOVEABLE_RESTRAINTS)}"
    restraint = ParticleInteraction((0, 0, 0), indexes)
    restraint.scale = 1000
    restraint.interaction_type = "spring"
    restraint.reset_velocities = True
    MOVEABLE_RESTRAINTS[key] = restraint

    with client.create_selection(key, indexes).modify() as selection:
        selection.renderer = RENDERER
        selection.velocity_reset = True
        selection.interaction_method = "group"
        SELECTIONS[key] = selection

    return key


def remove_restraint(key):
    disable_restraint(key)
    client.remove_selection(SELECTIONS[key])
    MOVEABLE_RESTRAINTS.pop(key, None)


def enable_restraint(key):
    if key in ACTIVE_RESTRAINTS:
        return
    ACTIVE_RESTRAINTS.add(key)
    positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions
    restraint = MOVEABLE_RESTRAINTS[key]
    restraint.position = positions[restraint.particles].sum(axis=0) / len(restraint.particles)
    imd.insert_interaction(key, restraint)


def disable_restraint(key):
    if key not in ACTIVE_RESTRAINTS:
        return
    ACTIVE_RESTRAINTS.remove(key)
    imd.remove_interaction(key)


def on_interaction_started(*, key: str, interaction: ParticleInteraction):
    if RESTRAINT_PREFIX in key:
        return
    MODE.on_interaction_started(key=key, interaction=interaction)


def on_interaction_stopped(*, key: str, interaction: ParticleInteraction):
    if RESTRAINT_PREFIX in key:
        return
    MODE.on_interaction_stopped(key=key, interaction=interaction)


imd_runner.app_server.imd.interaction_started.add_callback(on_interaction_started)
imd_runner.app_server.imd.interaction_stopped.add_callback(on_interaction_stopped)

## Commands

In [5]:
def clear_restraints():
    for restraint in MOVEABLE_RESTRAINTS:
        disable_restraint(restraint)
    MOVEABLE_RESTRAINTS.clear()

imd_runner.app_server.register_command("user/restraints/clear", clear_restraints)

In [6]:
from nanover.imd.imd_state import ParticleInteraction

class Mode:
    @classmethod
    def enter(cls):
        global MODE
        MODE = cls()

    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        pass

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        pass


MODE = Mode()

In [7]:
class InteractMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        interacted_indexes = set(interaction.particles)
        for key, restraint in MOVEABLE_RESTRAINTS.items():
            if interacted_indexes.intersection(restraint.particles):
                disable_restraint(key)

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        interacted_indexes = set(interaction.particles)
        for key, restraint in MOVEABLE_RESTRAINTS.items():
            if interacted_indexes.intersection(restraint.particles):
                enable_restraint(key)

imd_runner.app_server.register_command("user/restraints/interact", InteractMode.enter)

In [8]:
class ToggleMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        interacted_indexes = set(interaction.particles)
        removed = False
        for key, restraint in MOVEABLE_RESTRAINTS.items():
            if interacted_indexes.intersection(restraint.particles):
                remove_restraint(key)
                removed = True
        if not removed:
            add_restraint(interaction.particles)

imd_runner.app_server.register_command("user/restraints/toggle", ToggleMode.enter)

In [9]:
ToggleMode.enter()

## Recording

In [ ]:
from nanover.websocket.record import record_from_runner, BackgroundRecordingContext

RECORDING_COUNT = 0
RECORDER: BackgroundRecordingContext | None = None

def start_recording():
    global RECORDER, RECORDING_COUNT
    RECORDER = record_from_runner(imd_runner, f"RECORDING-{RECORDING_COUNT}-{imd_runner.simulation.name}.nanover.zip")
    RECORDING_COUNT += 1

def stop_recording():
    if RECORDER is not None:
        RECORDER.close()

imd_runner.app_server.register_command("user/recording/start", start_recording)
imd_runner.app_server.register_command("user/recording/stop", stop_recording)

